# Solutions · Chapter 03-07 · Vectors, distance, and shapes

E8 and E16 are the two to attempt first. E8 changes the answer without standardising anything, and
E16 measures something that quietly limits every distance-based method you will ever build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

flats = pd.DataFrame({
    "flat":  ["A", "B", "C", "D", "E"],
    "rent":  [1200, 1250, 2000, 1180, 1900],
    "rooms": [2, 4, 2, 4, 4],
    "km":    [1.0, 5.0, 1.0, 6.0, 1.5],
})
features = ["rent", "rooms", "km"]
X = flats[features].to_numpy(dtype=float)
wanted = np.array([1210.0, 4.0, 5.0])
print(flats.to_string(index=False))

## E1 · Why the raw distance chose the wrong flat

Because rent is measured in numbers that are hundreds of times larger than the other columns, so its
squared differences dominate the sum - a 40-euro gap contributes 1,600 while a two-room gap
contributes 4, and the formula has no way to know that the two rooms matter more.

## E2 · Which methods care about scale

**Scaling changes the answer:** k-nearest-neighbours, k-means, hierarchical clustering, PCA, support
vector machines with an RBF kernel, distance-based anomaly detection, and any regression fitted with
regularisation (Ridge and Lasso penalise coefficient sizes, so the units of each feature decide how
hard it is penalised).

**Scaling changes nothing:** decision trees, random forests, and gradient boosting. They split one
column at a time on thresholds, and a monotonic rescaling moves the threshold without changing which
rows land on each side. Plain linear regression without regularisation is also unaffected in its
predictions - the coefficients change to compensate exactly.

## E3 · What cosine ignores

**It ignores magnitude - the length of the vector - and compares only direction.**

**Right to ignore it:** comparing users by taste when some listen ten times as much as others;
comparing documents by topic when some are ten times longer. In both cases the total is a fact about
activity, not about the thing you are comparing.

**Wrong to ignore it:** comparing flats by rent and size, comparing patients by dosage and blood
pressure, comparing anything where "twice as much" is a real difference. A flat at 2,000 euros and one
at 200 euros with the same room ratio are not similar.

## E4 · Two distances by hand

In [ ]:
first, second = np.array([3.0, 4.0, 0.0]), np.array([0.0, 0.0, 12.0])
gap = first - second
print("differences        :", gap)
print("Euclidean (L2)     : sqrt(9 + 16 + 144) = sqrt(%.0f) = %.4f" % ((gap ** 2).sum(),
                                                                      np.sqrt((gap ** 2).sum())))
print("Manhattan (L1)     : 3 + 4 + 12 = %.4f" % np.abs(gap).sum())

**L2 is 13 and L1 is 19, so L1 is larger.**

**Will that always be true? Yes.** The L1 distance is always at least as large as the L2 distance, and
they are equal only when the difference lies along a single axis - when all but one of the gaps is
zero. It is the triangle inequality in a familiar form: walking along the streets is never shorter
than cutting across the block.

## E5 · A prediction, and a change of units

In [ ]:
weights = np.array([0.5, -2.0, 10.0])       # per m2, per year of age, for a garden
flat = np.array([80.0, 12.0, 1.0])
print("prediction: 80 x 0.5 + 12 x -2 + 1 x 10 = %.1f" % (flat @ weights))
print()
print("in square centimetres, 80 m2 becomes %.0f cm2" % (80 * 10_000))
print("so the area weight would have to become 0.5 / 10000 = %.7f to give the same prediction"
      % (0.5 / 10_000))

**The prediction is 26.0.**

**In square centimetres the area weight becomes 0.00005** - ten thousand times smaller - and every
prediction is identical. The model has not changed at all; only the units have.

This is 03-06's point in vector form, and it is the reason "the coefficient on area is tiny, so area
does not matter" is not an argument. It is also why regularised models *must* be given scaled inputs:
Ridge and Lasso penalise the size of the coefficients, so a feature measured in small units gets a
large coefficient and is punished for it, purely because of the unit chosen.

## E6 · Cosine by hand

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


u, v, w = np.array([4.0, 2.0]), np.array([1.0, 8.0]), np.array([8.0, 4.0])
print("[4,2] . [1,8] = %.0f ; lengths %.4f and %.4f" % (u @ v, np.linalg.norm(u), np.linalg.norm(v)))
print("cosine([4,2], [1,8]) = %.4f" % cosine(u, v))
print("cosine([4,2], [8,4]) = %.4f" % cosine(u, w))

**0.5547 and 1.0000.**

The second is exactly 1 because `[8, 4]` is `[4, 2]` doubled - the same direction, twice the length.
**Cosine similarity cannot tell a vector from any multiple of itself**, which is precisely the property
that makes it right for taste and wrong for quantity.

## E7 · `nearest`

In [ ]:
def nearest(X, query, scale=True):
    X = np.asarray(X, dtype=float)
    query = np.asarray(query, dtype=float)
    if scale:
        means, sds = X.mean(axis=0), X.std(axis=0)
        X, query = (X - means) / sds, (query - means) / sds
    return int(np.argmin(np.sqrt(((X - query) ** 2).sum(axis=1))))


print("nearest without scaling:", flats["flat"][nearest(X, wanted, scale=False)])
print("nearest with scaling   :", flats["flat"][nearest(X, wanted, scale=True)])

## E8 · The same data, rent in thousands

In [ ]:
X_thousands = X.copy()
X_thousands[:, 0] /= 1000
wanted_thousands = wanted.copy()
wanted_thousands[0] /= 1000

new_distances = np.sqrt(((X_thousands - wanted_thousands) ** 2).sum(axis=1))
print(pd.DataFrame({"flat": flats["flat"], "distance": new_distances.round(4)}).to_string(index=False))
print()
print("nearest with rent in euros     :", flats["flat"][nearest(X, wanted, scale=False)])
print("nearest with rent in thousands :", flats["flat"][int(new_distances.argmin())])

**Flat B, and nothing was standardised.**

**What this proves:** the "nearest" flat changed because somebody divided a column by a thousand. No
data was added, removed or corrected; the same five flats and the same requirements produced a
different answer.

So an unscaled distance is not a weak measurement of similarity - it is **not a measurement of
similarity at all**. It is a statement about which spreadsheet columns happen to be recorded in large
units. Standardising is not an improvement to the calculation; it is what makes the calculation mean
anything.

## E9 · A distance matrix without loops

In [ ]:
def distance_matrix(X):
    X = np.asarray(X, dtype=float)
    # (n, 1, d) against (1, n, d) broadcasts to (n, n, d)
    return np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=-1))


D = distance_matrix(X)
print(np.round(D, 1))

by_loop = np.array([[np.sqrt(((a - b) ** 2).sum()) for b in X] for a in X])
print("\nmatches the loop version:", np.allclose(D, by_loop))
print("diagonal is zero        :", np.allclose(np.diag(D), 0))
print("symmetric               :", np.allclose(D, D.T))
print()
for n in [1_000, 10_000, 50_000]:
    print("n = %6d  ->  %13d entries  =  %6.2f GB as float64" % (n, n * n, n * n * 8 / 1e9))

**Shape `(n, n)`, and for 10,000 rows that is 100 million entries and 0.8 GB.**

Worse, the intermediate `(n, n, d)` array that the broadcasting creates is `d` times larger again -
for three columns, 2.4 GB, and it is allocated before the sum reduces it. This is 01-03's 400 GB
broadcasting failure in a form you will actually reach for.

**What to do instead at scale:** compute distances in blocks of rows, use
`sklearn.metrics.pairwise_distances` with a `chunk_size`, or - much better - use a spatial index
(`KDTree`, `BallTree`) which finds the nearest neighbours without ever materialising the full matrix.
Those structures are how kNN scales, and E16 explains when they stop helping.

## E10 · The unscaled kNN on customer data

In [ ]:
customer_rng = np.random.default_rng(0)
customers = np.column_stack([
    customer_rng.uniform(18, 90, 2000),        # age
    customer_rng.uniform(15_000, 200_000, 2000),  # income
    customer_rng.uniform(0, 30, 2000),         # visits per month
])
query = customers.mean(axis=0)

squared = (customers - query) ** 2
share = (squared / squared.sum(axis=1, keepdims=True)).mean(axis=0)
print(pd.DataFrame({"column": ["age", "income", "visits_per_month"],
                    "typical range": ["18-90", "15,000-200,000", "0-30"],
                    "mean share of squared distance": share.round(6)}).to_string(index=False))

**Income decides every neighbour, contributing 99.94% of the squared distance on average.** Age
contributes 0.055% and visits 0.0017%.

So the "k nearest customers" are simply the k customers with the most similar income. Age and visit
frequency are in the model in the sense that they occupy memory, and in no other sense. The model
will appear to work - kNN on income alone is not useless - and nobody will discover that two of the
three features were never consulted.

**The prediction to make before running it:** the column with the largest numeric range wins, and with
ranges of roughly 72, 185,000 and 30, income's squared differences are around six orders of magnitude
larger. That estimate is available before any code is written, which is the point of the exercise.

## E11 · The recommender dominated by active users

**The mechanism:** with raw play counts, a very active user's vector is long - large numbers in every
component - so their Euclidean distance to everyone else is large, in every direction. They are far
from everybody and therefore nobody's neighbour, however closely their *taste* matches. Meanwhile two
inactive users are close to each other simply because both vectors are near the origin, regardless of
what they listen to.

Euclidean distance on raw counts is measuring **how much people listen** and treating that as taste.

**The fix: compare directions, not positions** - cosine similarity, or equivalently normalise each
user's vector to unit length before computing distances, which makes the two identical. A common
refinement is to log the counts first, so that the difference between 1 and 10 plays counts for more
than the difference between 500 and 509.

## E12 · `(5, 5)` instead of `(5,)`

In [ ]:
X_demo = np.zeros((5, 3))
print("X                 ", X_demo.shape)
print("X @ (3,)   gives  ", (X_demo @ np.zeros(3)).shape)
print("X @ (3, 5) gives  ", (X_demo @ np.zeros((3, 5))).shape, "  <- what they got")

**`weights` must have had shape `(3, 5)`.** The rule `(5, 3) @ (3, 5) -> (5, 5)` is the only way to
reach that output.

**What they most likely did:** built the weights from something with the wrong orientation - a
`DataFrame` column that came out two-dimensional, a `reshape` with the arguments the wrong way round,
or an array of five candidate weight-vectors that was meant to be looped over and got passed whole.

The tell is that the output is square with a side equal to the number of rows: **whenever a result is
`(n, n)` and you expected `(n,)`, something that should have been one vector is a matrix.**

## E13 · Why standardise

> "It puts every feature on a comparable scale, so that a distance or a penalty is not decided by
> which column happens to be measured in the largest units. It is essential for anything geometric -
> k-nearest-neighbours, k-means, PCA - and for regularised regression, where the penalty on a
> coefficient depends on the feature's units; on the flats data, unscaled distance gave 99.9% of its
> weight to rent and returned a flat with the wrong number of rooms. It is unnecessary for trees and
> forests, which split one column at a time and never compare across columns. What it silently assumes
> is that one standard deviation of each feature is equally important, which is a much better default
> than 'one euro equals one room' but is still an assumption, and where I have real domain knowledge I
> would weight deliberately instead. And it must be fitted on the training data alone - the mean and
> standard deviation come from the training set and are then applied unchanged to validation and test,
> because computing them over everything leaks information about the test set into training."

The last sentence is the one interviewers are actually listening for.

## E14 · Matching patients

**How to set up the distance:**

1. **Standardise all four columns** using the training cohort's means and standard deviations, so age
   in years and blood pressure in mmHg become comparable.
2. **Then apply deliberate weights**, because standardising has only made the features *comparable*,
   not *correctly prioritised*.

**The column I would weight more heavily: prior admissions.** For matching similar cases, a patient's
admission history carries more clinical information about their trajectory than a few mmHg of blood
pressure, and standardising alone would give it exactly the same influence as everything else.

**Why standardising alone cannot achieve it:** standardising equalises the *spread* of every column,
which is a statement about the data's variability, not about clinical importance. If blood pressure
happens to vary a lot in this cohort and admissions do not, standardising gives them equal weight and
in effect lets the cohort's variance decide what matters. Multiplying the standardised admissions
column by, say, 2 before computing distances is how you say "this is worth twice as much", and it
should be a documented, defended choice rather than a side effect.

Two further points worth raising: **BMI is itself derived from height and weight**, so including all
three would double-count; and **age may deserve a non-linear treatment**, since ten years matters far
more at 75 than at 35.

## E16 · How distance behaves as dimensions grow

In [ ]:
curse_rng = np.random.default_rng(1)

rows = []
for d in [2, 3, 5, 10, 20, 50, 100, 200]:
    points = curse_rng.random((500, d))
    queries = curse_rng.random((50, d))
    D = np.sqrt(((queries[:, None, :] - points[None, :, :]) ** 2).sum(axis=-1))
    rows.append({"dimensions": d,
                 "mean nearest": round(float(D.min(axis=1).mean()), 3),
                 "mean furthest": round(float(D.max(axis=1).mean()), 3),
                 "furthest / nearest": round(float((D.max(axis=1) / D.min(axis=1)).mean()), 3)})

curse = pd.DataFrame(rows)
print(curse.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(curse["dimensions"], curse["furthest / nearest"], "o-", color="#0072B2")
ax.axhline(1.0, color="#D55E00", linestyle="--", linewidth=1)
ax.text(120, 1.06, "1.0 = every point equally far", color="#D55E00", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("number of dimensions (log scale)")
ax.set_ylabel("furthest neighbour / nearest neighbour")
ax.set_title("In high dimensions, 'nearest' stops meaning much")
plt.tight_layout()
plt.show()

### The ratio collapses towards 1

In **2 dimensions** the furthest point is on average **74 times** further away than the nearest. By
**200 dimensions** it is **1.27 times** - the furthest point in the entire dataset is only 27% further
away than the closest one.

This is the **curse of dimensionality**, and the mechanism is worth understanding rather than
memorising. Distance in `d` dimensions is a sum of `d` squared differences. Each is a random quantity,
and adding many of them together makes the total concentrate around its average - the same
`1/sqrt(n)` effect from 03-02, applied to the coordinates of a point rather than to a sample. Every
pair of points ends up at nearly the same distance, because each pair's distance is an average over
many independent little differences.

**What it implies for nearest-neighbour methods:**

- **They degrade as features are added**, and not gracefully. With enough columns, "the nearest
  neighbour" is barely nearer than a random row, so the prediction is barely better than the
  population average.
- **Adding a feature is not free.** An uninformative column contributes noise to every distance and
  dilutes the informative ones. For kNN specifically, ten good features usually beat a hundred mixed
  ones.
- **Spatial indexes stop helping.** KDTree and BallTree rely on being able to rule out regions of
  space, and when everything is equidistant there is nothing to rule out - past roughly twenty
  dimensions they degenerate towards a full scan.
- **The usual responses** are dimensionality reduction before the distance (module 08), feature
  selection, learning a metric from the data, or choosing a method that does not depend on distance at
  all - which is one of the reasons trees and boosting are so often the practical choice on wide
  tabular data.

Real data is usually kinder than uniform random points, because real features are correlated and the
data lies on a lower-dimensional surface inside the space. That is what makes kNN work at all on
hundreds of columns - but the direction of the effect is always this one.

## Where to go next

**03-08 · Loss, gradients, and how a model is fitted.** You can now write a prediction as `X @ w`.
The final chapter of module 03 is about choosing `w` - by defining what "wrong" means and then walking
downhill, which is the same idea as 03-01's grid search for the number that minimised total error,
made efficient enough to work in a thousand dimensions.